In [12]:
import pandas as pd
import sys
from pathlib import Path

DATA_DIR = Path("../../data")

files = {
    "sanity": DATA_DIR / "FOCUS_Data_Export_TABLE_NW_SANITY.xlsx",
    "rxlev":  DATA_DIR / "FOCUS_Data_Export_TABLE_NW_RXLev.xlsx",
    "fh_rsl": DATA_DIR / "temp" / "export.xlsx",
    "rtwp":   DATA_DIR / "temp" / "export_optim.xlsx",
}

dfs = {}
dfs["sanity"] = pd.read_excel(files["sanity"], header=1)
dfs["rxlev"] = pd.read_excel(files["rxlev"], header=1)
dfs["fh_rsl"] = pd.read_excel(files["fh_rsl"])
dfs["rtwp"] = pd.read_excel(files["rtwp"])

# load the parsed MLO dataset we saved in step 4
dfs["atoll_links"] = pd.read_csv(Path("../data/processed/atoll_links.csv"))

sys.path.append(str(Path("../src").resolve()))

for name, df in dfs.items():
    print(name, df.shape)

sanity (6038, 14)
rxlev (6016, 6)
fh_rsl (1292, 8)
rtwp (295, 8)
atoll_links (3795, 37)


In [13]:
from preprocessing.numeric_cleaning import clean_numeric

test_values = [
    "700,77 m", "2 023,68 m", "-78,5 dBm", "41 Mbps", "0 Mbps",
    "53,81 dB", "-93.80 dBm", "34,9 dBi", None, "n/a (23 353,75 MHz)"
]

for v in test_values:
    print(repr(v), "->", clean_numeric(v))

'700,77 m' -> 700.77
'2 023,68 m' -> 2023.68
'-78,5 dBm' -> -78.5
'41 Mbps' -> 41.0
'0 Mbps' -> 0.0
'53,81 dB' -> 53.81
'-93.80 dBm' -> -93.8
'34,9 dBi' -> 34.9
None -> None
'n/a (23 353,75 MHz)' -> 23353.75


In [14]:
atoll_numeric_cols = [
    "capacity", "length_m",
    "bandwidth_mhz_end_a", "bandwidth_mhz_end_b",
    "antenna_size_m_end_a", "antenna_size_m_end_b",
    "antenna_height_m_end_a", "antenna_height_m_end_b",
    "antenna_direction_deg_end_a", "antenna_direction_deg_end_b",
    "antenna_gain_db_end_a", "antenna_gain_db_end_b",
    "tx_power_dbm_end_a", "tx_power_dbm_end_b",
    "tx_attenuator_db_end_a", "tx_attenuator_db_end_b",
    "rx_attenuator_db_end_a", "rx_attenuator_db_end_b",
    "threshold_dbm_end_a", "threshold_dbm_end_b",
    "main_rx_level_dbm_end_a", "main_rx_level_dbm_end_b",
    "threshold_degradation_db_end_a", "threshold_degradation_db_end_b",
    "main_flat_fade_margin_db_end_a", "main_flat_fade_margin_db_end_b",
]

df_atoll = dfs["atoll_links"].copy()

for col in atoll_numeric_cols:
    df_atoll[col] = df_atoll[col].apply(clean_numeric)

print(df_atoll[atoll_numeric_cols].dtypes)
print()
display(df_atoll[atoll_numeric_cols].describe())

capacity                          float64
length_m                          float64
bandwidth_mhz_end_a               float64
bandwidth_mhz_end_b               float64
antenna_size_m_end_a              float64
antenna_size_m_end_b              float64
antenna_height_m_end_a            float64
antenna_height_m_end_b            float64
antenna_direction_deg_end_a       float64
antenna_direction_deg_end_b       float64
antenna_gain_db_end_a             float64
antenna_gain_db_end_b             float64
tx_power_dbm_end_a                float64
tx_power_dbm_end_b                float64
tx_attenuator_db_end_a            float64
tx_attenuator_db_end_b            float64
rx_attenuator_db_end_a            float64
rx_attenuator_db_end_b            float64
threshold_dbm_end_a               float64
threshold_dbm_end_b               float64
main_rx_level_dbm_end_a           float64
main_rx_level_dbm_end_b           float64
threshold_degradation_db_end_a    float64
threshold_degradation_db_end_b    

,capacity,length_m,bandwidth_mhz_end_a,bandwidth_mhz_end_b,antenna_size_m_end_a,antenna_size_m_end_b,antenna_height_m_end_a,antenna_height_m_end_b,antenna_direction_deg_end_a,antenna_direction_deg_end_b,...,rx_attenuator_db_end_a,rx_attenuator_db_end_b,threshold_dbm_end_a,threshold_dbm_end_b,main_rx_level_dbm_end_a,main_rx_level_dbm_end_b,threshold_degradation_db_end_a,threshold_degradation_db_end_b,main_flat_fade_margin_db_end_a,main_flat_fade_margin_db_end_b
count,3795.000000,3795.000000,3793.000000,3793.000000,3795.000000,3795.000000,3795.000000,3795.000000,3795.000000,3795.000000,...,3795.000000,3795.000000,3795.000000,3795.000000,3795.000000,3795.000000,3795.000000,3794.000000,3788.000000,3788.000000
mean,177.817549,5479.921668,28.926443,28.926443,0.750588,0.745518,20.463881,28.402819,176.441447,179.718596,...,0.564480,0.564480,-69.883241,-69.883241,-31.140553,-31.243563,2.619499,2.621508,38.615396,38.725488
std,173.696951,6291.081727,28.807370,28.807370,0.521172,0.508118,8.926679,9.987314,103.445760,103.617577,...,1.250153,1.250153,10.516600,10.516600,6.221753,6.221452,0.785101,0.783511,13.418364,13.431373
min,0.000000,1.580000,2.000000,2.000000,0.250000,0.250000,0.000000,0.000000,0.000000,0.060000,...,0.000000,0.000000,-92.500000,-92.500000,-57.970000,-58.300000,1.000000,1.000000,7.130000,6.830000
25%,42.800000,1128.805000,23.000000,23.000000,0.300000,0.300000,16.000000,23.000000,87.215000,88.665000,...,0.000000,0.000000,-81.000000,-81.000000,-34.480000,-34.680000,3.000000,3.000000,29.267500,29.430000
50%,121.000000,2768.760000,23.000000,23.000000,0.600000,0.600000,17.000000,30.000000,174.720000,181.980000,...,0.000000,0.000000,-68.000000,-68.000000,-31.670000,-31.680000,3.000000,3.000000,35.450000,35.635000
75%,229.000000,7955.520000,38.000000,38.000000,1.200000,1.200000,25.000000,34.000000,265.040000,271.340000,...,0.600000,0.600000,-63.000000,-63.000000,-28.205000,-28.350000,3.000000,3.000000,48.502500,48.642500
max,2112.000000,64351.320000,500.000000,500.000000,2.400000,2.400000,120.000000,261.000000,359.990000,359.590000,...,15.000000,15.000000,-52.000000,-52.000000,8.600000,8.950000,3.000000,3.000000,99.450000,99.100000


In [15]:
import importlib
import preprocessing.numeric_cleaning
importlib.reload(preprocessing.numeric_cleaning)
from preprocessing.numeric_cleaning import clean_numeric, extract_bandwidth_mhz

In [16]:
from preprocessing.numeric_cleaning import extract_bandwidth_mhz

df_atoll["bandwidth_mhz_end_a"] = dfs["atoll_links"]["bandwidth_mhz_end_a"].apply(extract_bandwidth_mhz)
df_atoll["bandwidth_mhz_end_b"] = dfs["atoll_links"]["bandwidth_mhz_end_b"].apply(extract_bandwidth_mhz)

print(df_atoll[["bandwidth_mhz_end_a", "bandwidth_mhz_end_b"]].describe())

       bandwidth_mhz_end_a  bandwidth_mhz_end_b
count          3793.000000          3793.000000
mean             28.194898            28.194898
std              31.490604            31.490604
min               3.500000             3.500000
25%              14.000000            14.000000
50%              28.000000            28.000000
75%              28.000000            28.000000
max             500.000000           500.000000


In [17]:
print(open(Path("../src/preprocessing/numeric_cleaning.py")).read())

import re
import pandas as pd

def clean_numeric(value):
    """Convert a messy numeric string like '700,77 m', '-78,5 dBm', '41 Mbps'
    into a float. Returns None if it can't be parsed."""
    if pd.isna(value):
        return None
    if isinstance(value, (int, float)):
        return float(value)

    s = str(value).strip()

    # remove spaces used as thousands separators (e.g. "2 023,68")
    s = s.replace("\u00a0", " ")  # non-breaking space, sometimes used by Excel
    s = re.sub(r"(?<=\d) (?=\d)", "", s)  # remove spaces between digits

    # extract the numeric part: optional minus, digits, optional , or . decimal
    match = re.search(r"-?\d+(?:[.,]\d+)?", s)
    if not match:
        return None

    num_str = match.group(0).replace(",", ".")
    try:
        return float(num_str)
    except ValueError:
        return None


def extract_bandwidth_mhz(value):
    """Extract the true bandwidth from a channel code like
    'LK_38GHz_3.5MHz[-]' -> 3.5. Returns None if no MHz v

In [18]:
suspect = df_atoll[df_atoll["bandwidth_mhz_end_a"] == 500]
print("rows with 500 MHz bandwidth:", len(suspect))
display(dfs["atoll_links"].loc[suspect.index, ["_file", "band_end_a", "bandwidth_mhz_end_a"]])

rows with 500 MHz bandwidth: 3


,_file,band_end_a,bandwidth_mhz_end_a
119,ARI_0097_ARI_0050_Eband_H.xlsx,71-86 GHz,E_Band_500MHz[-]
120,ARI_0097_ARI_0050_Eband_V.xlsx,71-86 GHz,E_Band_500MHz[-]
1965,MAN_0022_TUN_0135_E-Band.xlsx,71-86 GHz,E_Band_500MHz[+]


In [19]:
sanity_numeric_cols = ["RSL (Min)", "RSL (Max)", "RSL (Avg)"]
rxlev_numeric_cols = ["Min RSL", "Avg RSL", "Max RSL"]

df_sanity = dfs["sanity"].copy()
df_rxlev = dfs["rxlev"].copy()

for col in sanity_numeric_cols:
    df_sanity[col] = df_sanity[col].apply(clean_numeric)

for col in rxlev_numeric_cols:
    df_rxlev[col] = df_rxlev[col].apply(clean_numeric)

print("--- sanity ---")
print(df_sanity[sanity_numeric_cols].dtypes)
display(df_sanity[sanity_numeric_cols].describe())

print("\n--- rxlev ---")
print(df_rxlev[rxlev_numeric_cols].dtypes)
display(df_rxlev[rxlev_numeric_cols].describe())

--- sanity ---
RSL (Min)    float64
RSL (Max)    float64
RSL (Avg)    float64
dtype: object


,RSL (Min),RSL (Max),RSL (Avg)
count,6035.000000,6035.000000,6035.000000
mean,-69.813687,-26.690141,-42.111597
std,26.116764,11.094708,11.137206
min,-99.900000,-98.700000,-98.940000
25%,-95.900000,-32.300000,-48.430000
50%,-76.200000,-28.900000,-40.730000
75%,-43.200000,-24.100000,-34.730000
max,0.000000,0.000000,0.000000



--- rxlev ---
Min RSL    float64
Avg RSL    float64
Max RSL    float64
dtype: object


,Min RSL,Avg RSL,Max RSL
count,6016.000000,6016.000000,6016.000000
mean,-70.034176,-42.276185,-26.785522
std,25.861083,10.858154,11.000148
min,-99.900000,-98.940000,-98.700000
25%,-95.900000,-48.460000,-32.300000
50%,-76.600000,-40.770000,-28.900000
75%,-43.375000,-34.790000,-24.200000
max,-17.800000,-9.310000,0.000000


In [20]:
import importlib
import preprocessing.numeric_cleaning
importlib.reload(preprocessing.numeric_cleaning)
from preprocessing.numeric_cleaning import flag_and_clean_sentinels

df_sanity = flag_and_clean_sentinels(df_sanity, ["RSL (Min)", "RSL (Max)", "RSL (Avg)"])
df_rxlev = flag_and_clean_sentinels(df_rxlev, ["Min RSL", "Avg RSL", "Max RSL"])

df_fh_rsl = dfs["fh_rsl"].copy()
df_fh_rsl = flag_and_clean_sentinels(df_fh_rsl, ["Max RSL"])
# RSL DIFF depends on Max RSL, so recompute it as NaN wherever Max RSL was a sentinel
df_fh_rsl.loc[df_fh_rsl["Max RSL"].isna(), "RSL DIFF"] = pd.NA

print("--- sanity sentinel counts ---")
print(df_sanity[["RSL (Min)_was_sentinel", "RSL (Max)_was_sentinel", "RSL (Avg)_was_sentinel"]].sum())

print("\n--- rxlev sentinel counts ---")
print(df_rxlev[["Min RSL_was_sentinel", "Avg RSL_was_sentinel", "Max RSL_was_sentinel"]].sum())

print("\n--- fh_rsl sentinel counts ---")
print(df_fh_rsl["Max RSL_was_sentinel"].sum())

print("\n--- sanity RSL columns after cleaning ---")
display(df_sanity[["RSL (Min)", "RSL (Max)", "RSL (Avg)"]].describe())

--- sanity sentinel counts ---
RSL (Min)_was_sentinel    992
RSL (Max)_was_sentinel    597
RSL (Avg)_was_sentinel     21
dtype: int64

--- rxlev sentinel counts ---
Min RSL_was_sentinel    973
Avg RSL_was_sentinel      2
Max RSL_was_sentinel    576
dtype: int64

--- fh_rsl sentinel counts ---
30

--- sanity RSL columns after cleaning ---


,RSL (Min),RSL (Max),RSL (Avg)
count,5043.000000,5438.000000,6014.000000
mean,-64.316260,-29.602115,-42.225811
std,24.405728,7.006057,10.853003
min,-98.400000,-98.000000,-97.910000
25%,-92.200000,-32.700000,-48.437500
50%,-59.600000,-29.500000,-40.760000
75%,-41.200000,-25.800000,-34.762500
max,-17.800000,-2.700000,-4.690000


In [21]:
df_sanity.to_csv(Path("../data/processed/sanity_clean.csv"), index=False)
df_rxlev.to_csv(Path("../data/processed/rxlev_clean.csv"), index=False)
df_fh_rsl.to_csv(Path("../data/processed/fh_rsl_clean.csv"), index=False)
dfs["rtwp"].to_csv(Path("../data/processed/rtwp_clean.csv"), index=False)
df_atoll.to_csv(Path("../data/processed/atoll_links_clean.csv"), index=False)

for name in ["sanity_clean", "rxlev_clean", "fh_rsl_clean", "rtwp_clean", "atoll_links_clean"]:
    p = Path("../data/processed") / f"{name}.csv"
    print(name, "->", p.exists(), p.stat().st_size, "bytes")

sanity_clean -> True 559725 bytes
rxlev_clean -> True 448522 bytes
fh_rsl_clean -> True 148013 bytes
rtwp_clean -> True 21001 bytes
atoll_links_clean -> True 1011141 bytes


# 02 — Preprocessing

**Goal:** Convert messy string columns (unit suffixes, French-comma decimals, compound
channel codes) into clean numeric types, and handle the sentinel-value pattern found
during EDA (0 dBm / ~-99 dBm placeholders for "no reading").

**Inputs:** raw `.xlsx` files + `data/processed/atoll_links.csv`

**Outputs:**
- `src/preprocessing/numeric_cleaning.py`:
  - `clean_numeric()` — generic unit/decimal cleanup
  - `extract_bandwidth_mhz()` — pulls real MHz value out of channel codes like `LK_38GHz_3.5MHz[-]`
  - `flag_and_clean_sentinels()` — replaces sentinel values with NaN, adds `_was_sentinel` flag columns
- `data/processed/sanity_clean.csv`, `rxlev_clean.csv`, `fh_rsl_clean.csv`, `rtwp_clean.csv`, `atoll_links_clean.csv`